# DWD 层 — 清洗统一 + 星型模型

ODS Yellow + Green → 统一字段口径 → 清洗 → 衍生列 → 写出 `fact_trips`

同时构建维度表：`dim_zone`、`dim_date`

## 1. SparkSession（启用动态分区覆盖）

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

spark = (
    SparkSession.builder
    .appName("TLC-DWD-Build")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    # 动态分区覆盖：INSERT OVERWRITE 只覆盖本次写入涉及的分区，保证幂等重跑
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE        = os.path.expanduser("~/tlc-spark-pipeline")
ODS_Y       = f"{BASE}/data/warehouse/ods/yellow"
ODS_G       = f"{BASE}/data/warehouse/ods/green"
DWD_FACT    = f"{BASE}/data/warehouse/dwd/fact_trips"
DWD_ZONE    = f"{BASE}/data/warehouse/dwd/dim_zone"
DWD_DATE    = f"{BASE}/data/warehouse/dwd/dim_date"

print(f"Spark {spark.version} ready | partitionOverwriteMode=dynamic")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/18 23:58:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.1 ready | partitionOverwriteMode=dynamic


## 2. 统一字段口径

黄色多 `Airport_fee`，绿色多 `trip_type` / `ehail_fee`。统一为相同列集合，差异字段补 null。

In [2]:
yellow = spark.read.parquet(ODS_Y)
green  = spark.read.parquet(ODS_G)

# 黄色：补绿色独有列
yellow_unified = (
    yellow
    .withColumn("trip_type",  F.lit(None).cast("int"))      # 绿色才有
    .withColumn("Airport_fee", F.col("Airport_fee"))         # 黄色已有
)

# 绿色：补黄色独有列，ehail_fee 全 null 不保留
green_unified = (
    green
    .withColumn("Airport_fee", F.lit(0.0).cast("double"))   # 绿色不去机场，补 0
    .drop("ehail_fee")                                       # 永远 null，丢弃
)

# 取两者共同列（按黄色顺序）
common_cols = [c for c in yellow_unified.columns if c in green_unified.columns]

combined = yellow_unified.select(common_cols).union(green_unified.select(common_cols))

print(f"Combined rows (pre-clean): {combined.count():,}")
print(f"Columns: {len(common_cols)} → {common_cols}")

Combined rows (pre-clean): 9,722,363
Columns: 24 → ['VendorID', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'source_type', 'ingestion_ts', 'pickup_year', 'pickup_month', 'trip_type']


## 3. 清洗 + 衍生列

In [3]:
from pyspark.sql.functions import col, unix_timestamp, round as spark_round, hour, month, year, when, dayofweek

cleaned = (
    combined
    .filter(col("fare_amount")    > 0)
    .filter(col("trip_distance")  > 0)
    .filter(col("dropoff_datetime") > col("pickup_datetime"))
    .filter(col("PULocationID").isNotNull())
    .filter(col("DOLocationID").isNotNull())
    .filter(col("passenger_count").between(1, 6))
    .filter(col("pickup_datetime") >= "2024-01-01")
    .filter(col("pickup_datetime") <  "2024-04-01")
)

fact = (
    cleaned
    .withColumn("trip_duration_min",
        spark_round((unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 60, 2))
    .withColumn("avg_speed_mph",
        spark_round(col("trip_distance") / (col("trip_duration_min") / 60), 2))
    .withColumn("fare_per_mile",
        spark_round(col("fare_amount") / col("trip_distance"), 2))
    # 金额口径统一：surcharge_total 汇总各类附加费
    .withColumn("surcharge_total",
        spark_round(
            F.coalesce(col("improvement_surcharge"), F.lit(0)) +
            F.coalesce(col("mta_tax"), F.lit(0)) +
            F.coalesce(col("congestion_surcharge"), F.lit(0)) +
            F.coalesce(col("Airport_fee"), F.lit(0)), 2))
    .withColumn("time_of_day",
        when(hour("pickup_datetime").between(6,  9),  "morning_rush")
        .when(hour("pickup_datetime").between(10, 15), "midday")
        .when(hour("pickup_datetime").between(16, 19), "evening_rush")
        .when(hour("pickup_datetime").between(20, 23), "night")
        .otherwise("late_night"))
    .withColumn("payment_label",
        when(col("payment_type") == 1, "credit_card")
        .when(col("payment_type") == 2, "cash")
        .when(col("payment_type") == 3, "no_charge")
        .otherwise("other"))
    .withColumn("pickup_year",  year("pickup_datetime"))
    .withColumn("pickup_month", month("pickup_datetime"))
    .withColumn("pickup_hour",  hour("pickup_datetime"))
    .withColumn("day_of_week",  dayofweek("pickup_datetime"))
    .filter(col("avg_speed_mph").between(0, 100))
    .filter(col("trip_duration_min").between(1, 180))
)

fact_count = fact.count()
print(f"fact_trips rows: {fact_count:,}")
fact.groupBy("source_type", "pickup_month").count().orderBy("source_type", "pickup_month").show()

26/10/18 23:58:40 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


fact_trips rows: 8,594,033


+-----------+------------+-------+
|source_type|pickup_month|  count|
+-----------+------------+-------+
|      green|           1|  48991|
|      green|           2|  46581|
|      green|           3|  51018|
|     yellow|           1|2713403|
|     yellow|           2|2709721|
|     yellow|           3|3024319|
+-----------+------------+-------+



## 4. 写出 fact_trips（动态分区覆盖）

In [4]:
%%time
# partitionOverwriteMode=dynamic：只覆盖本次涉及的分区，重跑某个月不影响其他月
(
    fact
    .write
    .partitionBy("source_type", "pickup_year", "pickup_month")
    .mode("overwrite")
    .parquet(DWD_FACT)
)
print(f"fact_trips written to: {DWD_FACT}")

26/10/18 23:58:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


fact_trips written to: /Users/lujiayi/tlc-spark-pipeline/data/warehouse/dwd/fact_trips
CPU times: user 26.1 ms, sys: 18.8 ms, total: 44.9 ms
Wall time: 31.3 s


## 5. 维度表：dim_zone

In [5]:
# 从 fact 中提取所有出现的 LocationID，构建区域维度表
pu_zones = fact.select(F.col("PULocationID").alias("location_id"))
do_zones = fact.select(F.col("DOLocationID").alias("location_id"))
dim_zone = pu_zones.union(do_zones).distinct().orderBy("location_id")

dim_zone.write.mode("overwrite").parquet(DWD_ZONE)
print(f"dim_zone: {dim_zone.count()} zones → {DWD_ZONE}")

dim_zone: 262 zones → /Users/lujiayi/tlc-spark-pipeline/data/warehouse/dwd/dim_zone


## 6. 维度表：dim_date

In [6]:
from pyspark.sql.functions import to_date, dayofweek, weekofyear, quarter

dim_date = (
    fact
    .select(to_date("pickup_datetime").alias("date"))
    .distinct()
    .withColumn("year",        year("date"))
    .withColumn("month",       month("date"))
    .withColumn("day",         F.dayofmonth("date"))
    .withColumn("day_of_week", dayofweek("date"))
    .withColumn("week",        weekofyear("date"))
    .withColumn("quarter",     quarter("date"))
    .withColumn("is_weekend",  (dayofweek("date").isin(1, 7)).cast("boolean"))
    .orderBy("date")
)

dim_date.write.mode("overwrite").parquet(DWD_DATE)
print(f"dim_date: {dim_date.count()} dates → {DWD_DATE}")
dim_date.show(5)

spark.stop()
print("DWD build complete.")

dim_date: 91 dates → /Users/lujiayi/tlc-spark-pipeline/data/warehouse/dwd/dim_date


+----------+----+-----+---+-----------+----+-------+----------+
|      date|year|month|day|day_of_week|week|quarter|is_weekend|
+----------+----+-----+---+-----------+----+-------+----------+
|2024-01-01|2024|    1|  1|          2|   1|      1|     false|
|2024-01-02|2024|    1|  2|          3|   1|      1|     false|
|2024-01-03|2024|    1|  3|          4|   1|      1|     false|
|2024-01-04|2024|    1|  4|          5|   1|      1|     false|
|2024-01-05|2024|    1|  5|          6|   1|      1|     false|
+----------+----+-----+---+-----------+----+-------+----------+
only showing top 5 rows



DWD build complete.
